# Grad-CAM + Adversarial Attack **Localization** (10-animal test)

This extends the earlier notebook. Now it can answer: **which part of the image was attacked?**

Two signals are used:
1. **Noise-magnitude map** = `|x_adv - x|` → the *ground truth* of where pixels were changed.
2. **Grad-CAM shift map** = `|CAM_after - CAM_before|` → where the model's *attention* moved.

The attack can be **global** (whole image) or **localized** (only a patch). The code auto-detects
which case it is and draws a bounding box around the attacked region, then we run it on **10 animals**.

> Uses a single GPU on purpose — one image at a time needs no multi-GPU. Runs fine on 2×T4.


In [ ]:
# ============================================================
# CELL 1 — Setup
# ============================================================
import torch, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision import models, transforms
from PIL import Image
import urllib.request, json, os

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

In [ ]:
# ============================================================
# CELL 2 — Pretrained model + readable class names
# ============================================================
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.eval().to(device)

url = "https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json"
idx2label = {int(k): v[1] for k, v in json.load(urllib.request.urlopen(url)).items()}

In [ ]:
# ============================================================
# CELL 3 — Preprocessing helpers
# Images stay in [0,1] pixel space; normalization happens inside the model call
# so the adversarial noise is measured in real pixels.
# ============================================================
SIZE = 224
to_tensor = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])

mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
normalize = lambda t: (t - mean) / std

def load_image(path_or_url):
    """Load a local path OR a URL into a [1,3,224,224] tensor in [0,1]."""
    if str(path_or_url).startswith("http"):
        fn = "/tmp/" + os.path.basename(path_or_url)
        if not os.path.exists(fn):
            urllib.request.urlretrieve(path_or_url, fn)
        path_or_url = fn
    img = Image.open(path_or_url).convert("RGB")
    return to_tensor(img).unsqueeze(0).to(device)

def to_np(t):                       # tensor [1,3,H,W] -> HWC numpy for plotting
    return t.squeeze().detach().cpu().permute(1, 2, 0).numpy()

def norm01(a):                      # scale array to 0..1 for display/threshold
    a = a.astype(np.float32)
    return (a - a.min()) / (a.max() - a.min() + 1e-8)

In [ ]:
# ============================================================
# CELL 4 — Grad-CAM (unchanged idea: weight last-conv feature maps by their
# gradient w.r.t. the chosen class, then overlay).
# ============================================================
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        layer.register_forward_hook(lambda m, i, o: setattr(self, "acts", o.detach()))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "grads", go[0].detach()))

    def __call__(self, inp, class_idx=None):
        out = self.model(normalize(inp))
        if class_idx is None:
            class_idx = out.argmax(1).item()
        self.model.zero_grad()
        out[0, class_idx].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)             # map importances
        cam = F.relu((w * self.acts).sum(1, keepdim=True))        # weighted sum
        cam = F.interpolate(cam, size=(SIZE, SIZE), mode="bilinear", align_corners=False)
        cam = norm01(cam.squeeze().cpu().numpy())
        return cam, class_idx, out.softmax(1)[0, class_idx].item()

cam_tool = GradCAM(model, model.layer4[-1])

In [ ]:
# ============================================================
# CELL 5 — FGSM attack with an OPTIONAL region mask
# mask=None -> attack the WHOLE image (global).
# mask set   -> noise is confined to that region only (localized).
# ============================================================
def fgsm_attack(inp, true_class, epsilon=0.03, mask=None):
    inp = inp.clone().detach().requires_grad_(True)
    out = model(normalize(inp))
    loss = F.cross_entropy(out, torch.tensor([true_class], device=device))
    model.zero_grad(); loss.backward()
    perturb = epsilon * inp.grad.sign()
    if mask is not None:
        perturb = perturb * mask            # zero-out noise outside the region
    return torch.clamp(inp + perturb, 0, 1).detach()

def random_box_mask(frac=0.22):
    """A random square region covering ~`frac` of the image. Returns (mask, box)."""
    side = int(SIZE * np.sqrt(frac))
    x0 = np.random.randint(0, SIZE - side); y0 = np.random.randint(0, SIZE - side)
    m = torch.zeros(1, 1, SIZE, SIZE, device=device)
    m[..., y0:y0+side, x0:x0+side] = 1.0
    return m, (x0, y0, x0+side, y0+side)

def smart_mask(cam_before, thresh=0.5):
    """Attack only where the model currently looks (Grad-CAM hotspot)."""
    m = torch.tensor(cam_before >= thresh, device=device, dtype=torch.float32)
    return m.view(1, 1, SIZE, SIZE), None      # true box is irregular -> None

def build_attack(kind, cam_before):
    """Returns (mask, true_box). kind in {'global','box','smart'}."""
    if kind == "global": return None, None
    if kind == "box":    return random_box_mask()
    if kind == "smart":  return smart_mask(cam_before)
    raise ValueError(kind)

In [ ]:
# ============================================================
# CELL 6 — Localize the attack from the noise-magnitude map
# Logic: if almost no pixels are near-zero -> noise is everywhere (GLOBAL).
#        otherwise the non-zero pixels form the attacked patch -> draw its box.
# ============================================================
def localize_attack(noise_mag, bg_frac=0.05, max_coverage=0.6):
    """noise_mag: [H,W] normalized 0..1. Returns (box|None, coverage). None = whole image."""
    if noise_mag.max() - noise_mag.min() < 1e-6:      # perfectly uniform
        return None, 1.0
    fg = noise_mag >= bg_frac                          # pixels above background
    coverage = float(fg.mean())
    if coverage > max_coverage:                        # spread across image -> global
        return None, coverage
    ys, xs = np.where(fg)
    return (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())), coverage

def iou(a, b):
    """Intersection-over-Union of two boxes; None if either is missing."""
    if a is None or b is None: return None
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1])
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix1-ix0) * max(0, iy1-iy0)
    area = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return round(inter/area, 2) if area > 0 else 0.0

In [ ]:
# ============================================================
# CELL 7 — Full per-image pipeline + visualization
# ============================================================
def draw_box(ax, box, color):
    if box is None: return
    x0, y0, x1, y1 = box
    ax.add_patch(patches.Rectangle((x0, y0), x1-x0, y1-y0,
                 fill=False, edgecolor=color, linewidth=2.5))

def run_one(path_or_url, attack="box", epsilon=0.05, show=True, name=""):
    x = load_image(path_or_url)

    # --- BEFORE: clean prediction + Grad-CAM
    cam_b, cls_b, conf_b = cam_tool(x)

    # --- ATTACK (whole image or a region)
    mask, true_box = build_attack(attack, cam_b)
    x_adv = fgsm_attack(x, cls_b, epsilon, mask)

    # --- AFTER: prediction + Grad-CAM on the adversarial image
    cam_a, cls_a, conf_a = cam_tool(x_adv)

    # --- WHERE was it attacked?  (two views)
    noise_mag = norm01((x_adv - x)[0].abs().mean(0).cpu().numpy())   # pixel-change map
    cam_delta = np.abs(cam_a - cam_b)                                 # attention-shift map
    det_box, coverage = localize_attack(noise_mag)
    region = "WHOLE IMAGE" if det_box is None else f"box {det_box}"
    fooled = cls_a != cls_b

    res = dict(name=name or os.path.basename(str(path_or_url)),
               before=idx2label[cls_b], after=idx2label[cls_a], fooled=fooled,
               attack=attack, coverage=round(coverage, 2),
               true_box=true_box, det_box=det_box, IoU=iou(true_box, det_box))

    if show:
        orig, advn = to_np(x), to_np(x_adv)
        fig, ax = plt.subplots(1, 5, figsize=(20, 4.2))
        ax[0].imshow(orig); draw_box(ax[0], true_box, "lime")
        ax[0].set_title(f"Original\n{res['before']} {conf_b:.0%}\n(green = true attack)")
        ax[1].imshow(noise_mag, cmap="hot"); draw_box(ax[1], det_box, "cyan")
        ax[1].set_title(f"|Noise| map\nDETECTED: {region}")
        ax[2].imshow(orig); ax[2].imshow(cam_b, cmap="jet", alpha=0.5)
        ax[2].set_title("Grad-CAM BEFORE")
        ax[3].imshow(advn); ax[3].imshow(cam_a, cmap="jet", alpha=0.5)
        ax[3].set_title(f"Grad-CAM AFTER\n{res['after']} {conf_a:.0%}")
        ax[4].imshow(cam_delta, cmap="jet"); draw_box(ax[4], det_box, "cyan")
        ax[4].set_title("Attention SHIFT\n|CAM after - before|")
        for a in ax: a.axis("off")
        flag = "FOOLED ✅" if fooled else "not fooled"
        fig.suptitle(f"{res['name']}  —  {res['before']} → {res['after']}   [{flag}]",
                     fontsize=13, y=1.02)
        plt.tight_layout(); plt.show()
    return res

## Quick demo on one image
A **localized** box attack — watch the noise map (panel 2) and attention-shift map (panel 5)
both light up only inside the attacked region, and the detected cyan box should sit on top of the
true green box.

In [ ]:
np.random.seed(0)
_ = run_one("https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n02129165_lion.JPEG",
            attack="box", epsilon=0.06)

In [ ]:
# ============================================================
# CELL 9 — Get 10 animal images
# Option A (recommended for your OWN images): attach a Kaggle dataset and set FOLDER.
# Option B (default): download 10 samples (needs Internet ON in Kaggle settings).
# ============================================================
FOLDER = ""   # e.g. "/kaggle/input/animals10/raw-img"  -> will sample 10 images from here

if FOLDER and os.path.isdir(FOLDER):
    exts = (".jpg", ".jpeg", ".png")
    paths = []
    for root, _, files in os.walk(FOLDER):
        for f in files:
            if f.lower().endswith(exts): paths.append(os.path.join(root, f))
    np.random.seed(1)
    images = list(np.random.choice(paths, size=min(10, len(paths)), replace=False))
else:
    base = "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/"
    images = [base + n for n in [
        "n02099601_golden_retriever.JPEG", "n02123045_tabby.JPEG",
        "n02391049_zebra.JPEG",            "n02129165_lion.JPEG",
        "n02129604_tiger.JPEG",            "n02510455_giant_panda.JPEG",
        "n01518878_ostrich.JPEG",          "n01806143_peacock.JPEG",
        "n01882714_koala.JPEG",            "n02007558_flamingo.JPEG"]]
print(f"{len(images)} images ready")

In [ ]:
# ============================================================
# CELL 10 — Run the pipeline on all 10 (localized 'box' attacks)
# Change attack="global" for whole-image, or "smart" to attack the Grad-CAM hotspot.
# ============================================================
np.random.seed(7)               # reproducible random boxes
ATTACK, EPS = "box", 0.06
results = [run_one(p, attack=ATTACK, epsilon=EPS, show=True) for p in images]

In [ ]:
# ============================================================
# CELL 11 — Summary table
# IoU = overlap between the TRUE attacked region and the DETECTED region (1.0 = perfect).
# ============================================================
import pandas as pd
df = pd.DataFrame(results)[
    ["name", "before", "after", "fooled", "attack", "coverage", "IoU", "true_box", "det_box"]]
fooled_rate = df["fooled"].mean()
mean_iou = df["IoU"].dropna().mean() if df["IoU"].notna().any() else float("nan")
print(f"Fooled: {fooled_rate:.0%} of images   |   Mean localization IoU: {mean_iou:.2f}")
df

## How to read the results

- **Noise map (panel 2)** = literally where pixels were changed. The **cyan box** is the auto-detected
  attacked region; for `attack="box"` it should overlap the **green** true region (high **IoU**).
- **Attention shift (panel 5)** = `|CAM_after - CAM_before|`, i.e. where the attack moved the model's focus.
  Even a small localized patch can swing the global prediction.
- **Summary IoU** tells you, on average, how accurately the pipeline located the attacked part.

### Try next
- `attack="global"` → noise everywhere, detector reports **WHOLE IMAGE** (IoU is N/A).
- `attack="smart"`  → noise only on the Grad-CAM hotspot; fools the model with the least perturbation.
- Raise `EPS` for a stronger attack (higher fool rate, more visible noise).
- Point `FOLDER` at a Kaggle animals dataset (e.g. `animals10`) to test your own images.
